In [10]:
from preprocess_data import *

In [18]:
# preprocess_data.py

import os
import pickle
import click
import pandas as pd
import requests
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder


def dump_pickle(obj, filename: str):
    with open(filename, "wb") as f_out:
        return pickle.dump(obj, f_out)



def preprocess(metadata: pd.DataFrame, sc: StandardScaler, ohe: OneHotEncoder, fit_dv : bool = False):
    
    time_columns = ['Batch_OGD_date','Batch_OGD_heure_debut','Batch_OGD_heure_fin']
    timed = metadata[time_columns]
    for col in timed.columns:
        timed.loc[:,col] = pd.to_datetime(timed[col])

    timed['year'] = pd.DatetimeIndex(timed['Batch_OGD_date']).year.astype(str)
    timed['month'] = pd.DatetimeIndex(timed['Batch_OGD_date']).month.astype(str)
    timed['time_exfo'] = (timed['Batch_OGD_heure_fin'] - timed['Batch_OGD_heure_debut']) / pd.Timedelta(hours=1)
    metadata_timed = pd.concat([metadata,timed],axis=1)

    numerical_columns = ['Batch_OGD_KC8_masse',
                        'Batch_OGD_Temperature',
                        'Batch_OGD_Agitation',
                        'Batch_OGD_room_HR',
                        'Batch_OGD_room_T',
                        'time_exfo']
    categorical_columns = ['Batch_OGD_Technicien',
                            'Batch_OGD_KC8_batch',
                            'Batch_OGD_THF_batch',
                            'year',
                            'month']

    if fit_dv:
        scaled = sc.fit_transform(metadata_timed[numerical_columns])
        scaled = pd.DataFrame(scaled, columns=numerical_columns)
        onehotencoded = ohe.fit_transform(metadata_timed[categorical_columns])
        onehotencoded = pd.DataFrame(onehotencoded, columns = [a for b in ohe.categories_ for a in b])
    else:
        scaled = sc.transform(metadata_timed[numerical_columns])
        scaled = pd.DataFrame(scaled, columns=numerical_columns)
        onehotencoded = ohe.transform(metadata_timed[categorical_columns])
        onehotencoded = pd.DataFrame(onehotencoded, columns = [a for b in ohe.categories_ for a in b])

    metadata_ready = pd.concat([scaled,onehotencoded],axis=1)

    return metadata_ready, sc, ohe


def run_data_prep(raw_data_path: str, dest_path: str, dataset: str = "green"):


    # raw_data_path = 'ML_Models/data'


    with open(f'{raw_data_path}/train.pkl', 'rb') as file: 
        df_train = pickle.load(file) 
    with open(f'{raw_data_path}/test.pkl', 'rb') as file: 
        df_test = pickle.load(file) 


    # Fit the kmeans and preprocess data
    sc = StandardScaler()
    ohe = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
    X_train, sc, ohe = preprocess(df_train, sc, ohe, fit_dv=True)
    X_test, _, _ = preprocess(df_test, sc, ohe, fit_dv=False)

    # Create dest_path folder unless it already exists
    os.makedirs(dest_path, exist_ok=True)

    # Save DictVectorizer and datasets
    dump_pickle(sc, os.path.join(dest_path, "sc.pkl"))
    dump_pickle(ohe, os.path.join(dest_path, "ohe.pkl"))
    dump_pickle((X_train), os.path.join(dest_path, "train_.pkl"))
    dump_pickle((X_test), os.path.join(dest_path, "test_.pkl"))



In [19]:
run_data_prep("data","data/output")

Index(['Batch_OGD_date', 'Batch_OGD_Technicien', 'Batch_OGD_KC8_batch',
       'Batch_OGD_KC8_masse', 'Batch_OGD_THF_batch', 'Batch_OGD_Temperature',
       'Batch_OGD_Agitation', 'Batch_OGD_heure_debut', 'Batch_OGD_heure_fin',
       'Batch_OGD_room_HR', 'Batch_OGD_room_T', 'Batch_OGD_Analyses',
       'Batch_OGD_date', 'Batch_OGD_heure_debut', 'Batch_OGD_heure_fin',
       'year', 'month', 'time_exfo'],
      dtype='object')
Index(['Batch_OGD_date', 'Batch_OGD_Technicien', 'Batch_OGD_KC8_batch',
       'Batch_OGD_KC8_masse', 'Batch_OGD_THF_batch', 'Batch_OGD_Temperature',
       'Batch_OGD_Agitation', 'Batch_OGD_heure_debut', 'Batch_OGD_heure_fin',
       'Batch_OGD_room_HR', 'Batch_OGD_room_T', 'Batch_OGD_Analyses',
       'Batch_OGD_date', 'Batch_OGD_heure_debut', 'Batch_OGD_heure_fin',
       'year', 'month', 'time_exfo'],
      dtype='object')


/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_28208/1177375958.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  timed['year'] = pd.DatetimeIndex(timed['Batch_OGD_date']).year.astype(str)
/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_28208/1177375958.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  timed['month'] = pd.DatetimeIndex(timed['Batch_OGD_date']).month.astype(str)
/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_28208/1177375958.py:27: SettingWithCopyWarn

In [14]:
# train.py

import os
import pickle
import click

from sklearn.cluster import KMeans
from sklearn.metrics import rand_score

##############
import mlflow

##############


#####################
region = "France Central" # "West Europe"
subscription_id = "974386b8-dfe6-43cc-94af-17335974d64a"
resource_group = "mlops-promo"
workspace = "mlops-workspace-promo"
# azureml_mlflow_uri = f"azureml://{region}.api.azureml.ms/mlflow/v1.0/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces/{workspace_name}"
# mlflow.set_tracking_uri(azureml_mlflow_uri)
credentials = DefaultAzureCredential()
ml_client = MLClient(
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    credential=credentials,)
ws = ml_client.workspaces.get(name=workspace)
####################

# mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_tracking_uri = ws.mlflow_tracking_uri
mlflow.set_experiment(experiment_name="k-means-groups")

def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./data",
    help="Location where the processed data was saved"
)
def run_train(data_path: str):


    ############################
    mlflow.sklearn.autolog()
    #####################


    X_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_test = load_pickle(os.path.join(data_path, "val.pkl"))




    with mlflow.start_run():
        for i in range(1,11):
            mlflow.log_param("groups_nbr", i)

            kmeans = KMeans(n_clusters=i, random_state=0, n_init="auto")
            w1 = kmeans.fit_transform(X_train) # metadata_ready.iloc[:120,1:]
            w2 = kmeans.fit_transform(X_test) # metadata_ready.iloc[40:,1:]

            rand_score = rand_score(w1[:-40], w2[:40])
            mlflow.log_metric("adj_rand_score", rand_score)


if __name__ == '__main__':
    run_train()




ohe.pkl     sc.pkl      test.pkl    test_.pkl   train.pkl   train_.pkl
